# Uniform-goal baseline

This notebook introduces the BoxGPT environment. The sensor repeatedly moves toward a uniformly sampled goal, samples a new goal when it arrives, and stops when the candidate rectangles have sufficiently low variance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from box_gym import BoxGym, action_toward

The action is a two-dimensional velocity. Measurements are triples `(x, y, label)`, where label 1 means the sampled point lies inside the hidden rectangle. Candidate boxes have columns `(left, bottom, right, top)`.

In [ ]:
env = BoxGym(
    sensor_box_size=0.12,
    num_sensor_samples=4,
    max_velocity=0.25,
    inference_num=100,
)
obs, info = env.reset(seed=11)

print("Observation keys:", tuple(obs))
print("Action space:", env.action_space)
print("Initial uncertainty:", round(info["uncertainty"], 5))

fig, ax = plt.subplots(figsize=(6, 6))
env.plot(ax)
plt.show()

In [ ]:
rng = np.random.default_rng(7)
margin = env.sensor_box_size / 2
goal = rng.uniform(margin, 1.0 - margin, size=2)
trajectory = [obs["sensor_pos"].copy()]
goals_visited = 0

for step in range(300):
    if np.linalg.norm(obs["sensor_pos"] - goal) < 0.025:
        goal = rng.uniform(margin, 1.0 - margin, size=2)
        goals_visited += 1

    action = action_toward(obs["sensor_pos"], goal, env.max_velocity)
    obs, reward, terminated, truncated, info = env.step(action)
    trajectory.append(obs["sensor_pos"].copy())
    if terminated or truncated:
        break

print(f"Ran {step + 1} steps and reached {goals_visited} goals")
print(f"Final uncertainty: {info['uncertainty']:.5f}")

fig, ax = plt.subplots(figsize=(7, 7))
env.plot(ax, goal=goal, trajectory=np.asarray(trajectory))
ax.set_title("Uniform-goal baseline")
plt.show()
env.close()